In [2]:
import pandas as pd
from nselib import capital_market
from tqdm import tqdm
import os

print("Downloading NSE Equity List...")

equities = capital_market.equity_list()

df = pd.DataFrame(equities)

print(df.head())

df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

# Rename important columns
rename_map = {
    "symbol": "symbol",
    "name_of_company": "company_name",
    "series": "series",
    "date_of_listing": "listing_date",
    "isin_number": "isin"
}

df = df.rename(columns=rename_map)

# Keep useful columns
cols = [
    "symbol",
    "company_name",
    "series",
    "listing_date"
]

df = df[cols]

# Status
df["status"] = "ACTIVE"

# Placeholder columns
df["industry"] = None
df["sector"] = None
df["market_cap"] = None
df["bse_code"] = None
df["website"] = None

# Aliases
def aliases(row):

    company = row["company_name"]

    aliases = set()

    aliases.add(company)

    aliases.add(company.replace("Limited","").strip())

    aliases.add(company.replace("Ltd.","").strip())

    aliases.add(company.replace("Ltd","").strip())

    aliases.add(company.replace("LIMITED","").strip())

    aliases.add(row["symbol"])

    return list(aliases)

df["aliases"] = df.apply(aliases, axis=1)

print(df.head())

os.makedirs("data/processed", exist_ok=True)

df.to_csv("data/processed/companies_master_1.csv", index=False)

df.to_parquet("data/processed/companies_master.parquet", index=False)

print(f"\nSaved {len(df)} companies.")

       SYMBOL                           NAME OF COMPANY  SERIES  \
0   20MICRONS                        20 Microns Limited      EQ   
1  21STCENMGM  21st Century Management Services Limited      EQ   
2      360ONE                       360 ONE WAM LIMITED      EQ   
3  3BBLACKBIO                    3B Blackbio Dx Limited      EQ   
4   3IINFOLTD                       3i Infotech Limited      EQ   

   DATE OF LISTING   FACE VALUE  
0      06-OCT-2008            5  
1      03-MAY-1995           10  
2      19-SEP-2019            1  
3      20-APR-2026           10  
4      22-OCT-2021           10  
       symbol                              company_name series listing_date  \
0   20MICRONS                        20 Microns Limited     EQ  06-OCT-2008   
1  21STCENMGM  21st Century Management Services Limited     EQ  03-MAY-1995   
2      360ONE                       360 ONE WAM LIMITED     EQ  19-SEP-2019   
3  3BBLACKBIO                    3B Blackbio Dx Limited     EQ  20-APR-2026  

In [ ]:
import pandas as pd
from nselib import capital_market
from tqdm import tqdm
import yfinance as yf
import os

print("Downloading NSE Equity List...")
equities = capital_market.equity_list()
df = pd.DataFrame(equities)

# Clean column names
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

print("Raw columns from NSE:", df.columns.tolist())

# Rename important columns
rename_map = {
    "symbol": "symbol",
    "name_of_company": "company_name",
    "series": "series",
    "date_of_listing": "listing_date",
    "isin_number": "isin",
    "isin": "isin",           # In case they already name it ISIN
    "isin_code": "isin"       # In case they renamed it to ISIN CODE
}

df = df.rename(columns=rename_map)

# Keep useful columns
cols = ["symbol", "company_name", "series", "listing_date", "isin"]

# This intersects our desired columns with the columns that actually exist in the dataframe
valid_cols = [c for c in cols if c in df.columns]
df = df[valid_cols]

if "isin" not in df.columns:
    print("WARNING: ISIN was not found in NSE data. Filling with blanks.")
    df["isin"] = None

# Status
df["status"] = "ACTIVE"

# ---------------------------------------------------------
# NEW LOGIC: Fetching Missing Details via yfinance
# ---------------------------------------------------------
print("Fetching Industry, Sector, Market Cap, and Website from yfinance...")

# Create empty lists to store the fetched data
industries = []
sectors = []
market_caps = []
websites = []

# Loop through each symbol with a progress bar
for symbol in tqdm(df['symbol']):
    yf_symbol = f"{symbol}.NS" # NSE symbols need the .NS extension
    
    try:
        # Fetch the ticker info
        info = yf.Ticker(yf_symbol).info
        
        # Extract required fields (use .get() to avoid KeyErrors if data is missing)
        industries.append(info.get('industry', None))
        sectors.append(info.get('sector', None))
        market_caps.append(info.get('marketCap', None))
        websites.append(info.get('website', None))
        
    except Exception:
        # If the ticker isn't found or an error occurs, append None
        industries.append(None)
        sectors.append(None)
        market_caps.append(None)
        websites.append(None)

# Add the new columns to your dataframe
df['industry'] = industries
df['sector'] = sectors
df['market_cap'] = market_caps
df['website'] = websites

# Note on BSE Code: yfinance does not reliably provide BSE codes for NSE stocks.
# We will leave it empty for now, or you can map it later.
df['bse_code'] = None

# ---------------------------------------------------------
# Aliases
# ---------------------------------------------------------
def aliases(row):
    company = str(row["company_name"])
    aliases = set()
    aliases.add(company)
    aliases.add(company.replace("Limited","").strip())
    aliases.add(company.replace("Ltd.","").strip())
    aliases.add(company.replace("Ltd","").strip())
    aliases.add(company.replace("LIMITED","").strip())
    aliases.add(str(row["symbol"]))
    return list(aliases)

df["aliases"] = df.apply(aliases, axis=1)

print(df.head())

os.makedirs("data/processed", exist_ok=True)
df.to_csv("data/processed/companies_master_1.csv", index=False)
df.to_parquet("data/processed/companies_master.parquet", index=False)

print(f"\nSaved {len(df)} companies.")

Raw columns from NSE: ['symbol', 'name_of_company', 'series', 'date_of_listing', 'face_value']
Fetching Industry, Sector, Market Cap, and Website from yfinance...


 42%|████▏     | 993/2382 [06:32<07:38,  3.03it/s]Exception ignored from cffi callback <function buffer_callback at 0x000001CB4A5FB880>:
Traceback (most recent call last):
  File "c:\Users\chandra.kiran\AppData\Local\anaconda3\envs\tinku\Lib\site-packages\curl_cffi\curl.py", line 125, in buffer_callback
    @ffi.def_extern()
    
KeyboardInterrupt: 
 44%|████▍     | 1047/2382 [06:54<09:21,  2.38it/s]

In [1]:
import asyncio
import os
import aiohttp
import pandas as pd
from nselib import capital_market
from tqdm.asyncio import tqdm

# We need a custom header so Yahoo Finance doesn't block the async requests
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

async def fetch_ticker_info(session, symbol, semaphore):
    """Fetches a single stock's metadata asynchronously from Yahoo Finance."""
    async with semaphore:
        url = f"https://query2.finance.yahoo.com/v10/finance/quoteSummary/{symbol}.NS?modules=assetProfile,summaryDetail"
        try:
            async with session.get(url, headers=HEADERS, timeout=10) as response:
                if response.status != 200:
                    return symbol, None, None, None, None
                
                data = await response.json()
                result = data.get('quoteSummary', {}).get('result', [{}])[0]
                
                # Extract details safely from the json modules
                profile = result.get('assetProfile', {})
                detail = result.get('summaryDetail', {})
                
                industry = profile.get('industry', None)
                sector = profile.get('sector', None)
                website = profile.get('website', None)
                
                # Market cap is deeply nested in yfinance's raw json payload
                market_cap = detail.get('marketCap', {}).get('raw', None)
                
                return symbol, industry, sector, market_cap, website
        except Exception:
            return symbol, None, None, None, None

async def download_all_metadata(symbols):
    """Manages the parallel downloading pool."""
    # Limit to 30 concurrent requests so Yahoo Finance doesn't rate-limit us
    sem = asyncio.Semaphore(30) 
    
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_ticker_info(session, sym, sem) for sym in symbols]
        
        # Wrap with tqdm.gather for a live async progress bar
        results = await tqdm.gather(*tasks, desc="Fetching data concurrently")
        return results

def main():
    print("Downloading NSE Equity List...")
    equities = capital_market.equity_list()
    df = pd.DataFrame(equities)

    # Clean column names
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

    # Rename important columns safely
    rename_map = {
        "symbol": "symbol",
        "name_of_company": "company_name",
        "series": "series",
        "date_of_listing": "listing_date",
    }
    df = df.rename(columns=rename_map)

    # Keep useful columns safely
    cols = ["symbol", "company_name", "series", "listing_date"]
    valid_cols = [c for c in cols if c in df.columns]
    df = df[valid_cols]

    # Add default placeholders
    df["status"] = "ACTIVE"
    df["isin"] = None
    df['bse_code'] = None

    # Run the async downloader framework
    # (If running inside Jupyter Notebook, use: results = await download_all_metadata(df['symbol'].tolist()))
    try:
        # For standard python scripts
        loop = asyncio.get_event_loop()
        results = loop.run_until_complete(download_all_metadata(df['symbol'].tolist()))
    except RuntimeError:
        # For Jupyter Notebook environments where the loop is already running
        import nest_asyncio
        nest_asyncio.apply()
        results = asyncio.run(download_all_metadata(df['symbol'].tolist()))

    # Map the results back into a lookup dictionary
    mapping = {res[0]: {"industry": res[1], "sector": res[2], "market_cap": res[3], "website": res[4]} for res in results}

    # Populate the main dataframe
    df['industry'] = df['symbol'].map(lambda x: mapping[x]['industry'])
    df['sector'] = df['symbol'].map(lambda x: mapping[x]['sector'])
    df['market_cap'] = df['symbol'].map(lambda x: mapping[x]['market_cap'])
    df['website'] = df['symbol'].map(lambda x: mapping[x]['website'])

    # Aliases Generator
    def aliases(row):
        company = str(row["company_name"])
        aliases_set = set()
        aliases_set.add(company)
        aliases_set.add(company.replace("Limited","").strip())
        aliases_set.add(company.replace("Ltd.","").strip())
        aliases_set.add(company.replace("Ltd","").strip())
        aliases_set.add(company.replace("LIMITED","").strip())
        aliases_set.add(str(row["symbol"]))
        return list(aliases_set)

    df["aliases"] = df.apply(aliases, axis=1)

    # Save output
    os.makedirs("data/processed", exist_ok=True)
    df.to_csv("data/processed/companies_master_1.csv", index=False)
    df.to_parquet("data/processed/companies_master.parquet", index=False)

    print(f"\nSaved {len(df)} companies successfully!")

if __name__ == "__main__":
    main()

Fetching data concurrently: 100%|██████████| 2382/2382 [00:13<00:00, 182.38it/s]
C:\Users\chandra.kiran\AppData\Local\Temp\ipykernel_42780\3116942233.py:89: RuntimeWarning: coroutine 'download_all_metadata' was never awaited
  results = asyncio.run(download_all_metadata(df['symbol'].tolist()))



Saved 2382 companies successfully!


In [2]:
import pandas as pd
import requests
import json
import time
from tqdm import tqdm
import os

# Load your existing master file (change filename to match yours)
INPUT_FILE = "data/processed/companies_master_1.csv"
OUTPUT_FILE = "data/processed/companies_final_with_industry.csv"

df = pd.read_csv(INPUT_FILE)

# Headers to pretend we are a standard web browser so we don't get blocked
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

print(f"Starting extraction for {len(df)} companies using Screener database...")

sectors = []
industries = []

for index, row in tqdm(df.iterrows(), total=len(df)):
    symbol = str(row['symbol']).strip()
    
    # 1. Query Screener's public autocomplete to get the exact profile URL path
    search_url = f"https://www.screener.in/api/company/search/?q={symbol}"
    
    sector_found = None
    industry_found = None
    
    try:
        response = requests.get(search_url, headers=HEADERS, timeout=5)
        if response.status_code == 200:
            results = response.json()
            
            # If search results are returned, pick the best matching company page
            if results:
                # Find exact or first match
                company_path = results[0]['url']
                
                # 2. Scrape the specific company page info JSON data
                # Adding a trailing slash to the company path for clean URL resolution
                if not company_path.endswith('/'):
                    company_path += '/'
                company_url = f"https://www.screener.in{company_path}"
                
                comp_response = requests.get(company_url, headers=HEADERS, timeout=5)
                if comp_response.status_code == 200:
                    html_text = comp_response.text
                    
                    # Screener conveniently hides the sector/industry inside a text block we can parse
                    if 'Sector:' in html_text:
                        # Extract Sector
                        sector_part = html_text.split('Sector:')[1].split('\n')[0]
                        sector_found = sector_part.replace('</a>', '').split('>')[-1].strip()
                    
                    if 'Industry:' in html_text:
                        # Extract Industry
                        industry_part = html_text.split('Industry:')[1].split('\n')[0]
                        industry_found = industry_part.replace('</a>', '').split('>')[-1].strip()
                        
    except Exception as e:
        # If an error happens, we pass and keep it None
        pass
        
    sectors.append(sector_found)
    industries.append(industry_found)
    
    # Tiny pause (0.1 seconds) to be respectful to the server and prevent IP bans
    time.sleep(0.1)

# Map the fetched data back to your original DataFrame
df['sector'] = sectors
df['industry'] = industries

# Save the final file
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
df.to_csv(OUTPUT_FILE, index=False)

print(f"\nProcessing complete! Successfully filled rows and saved to {OUTPUT_FILE}")
print(df[['symbol', 'company_name', 'sector', 'industry']].head(10))

Starting extraction for 2382 companies using Screener database...


100%|██████████| 2382/2382 [35:15<00:00,  1.13it/s] 


Processing complete! Successfully filled rows and saved to data/processed/companies_final_with_industry.csv
       symbol                              company_name sector industry
0   20MICRONS                        20 Microns Limited   None     None
1  21STCENMGM  21st Century Management Services Limited   None     None
2      360ONE                       360 ONE WAM LIMITED   None     None
3  3BBLACKBIO                    3B Blackbio Dx Limited   None     None
4   3IINFOLTD                       3i Infotech Limited   None     None
5     3MINDIA                          3M India Limited   None     None
6      3PLAND                  3P Land Holdings Limited   None     None
7      5PAISA                    5Paisa Capital Limited   None     None
8     63MOONS             63 moons technologies limited   None     None
9    A2ZINFRA             A2Z Infra Engineering Limited   None     None


In [3]:
pip install pyahocorasick pandas

Note: you may need to restart the kernel to use updated packages.
